The knapsack problem
====================

**Author:** Rafael



## Inicio



Nuestro punto de partida es el código que genera sucesiones de 0's y 1's por medio de backtracking.



In [1]:
def knapsack(l, X=None):
    if X is None:
        X = []

    if l == 4:
        yield X[:] 
    else:
        choices = [0, 1] 
        
        # Loop through each choice
        for choice in choices:
            X.append(choice)             # 1. Make the choice
            yield from knapsack(l+1, X)  # 2. Explore that path
            X.pop()                      # 3. Backtrack (undo the choice)

gens = knapsack(0)
list(gens)

## Problema de la mochila



Supongamos que tenemos objetos, respectivamente con pesos: 5, 7, 3, 9 (en kilogramos) y valor: 6, 4, 5, 7 (en pesos) Tenemos una mochila con capacidad para 16 kilos. ¿Qué objetos podemos llevar en la mochila de tal manera que el valor total de los objetos en la mochila sea el máximo?

Representaremos a los pesos y los valores con dos listas, además necesitamos la capacidad máxima:



In [1]:
weights = [5, 7, 3, 9]
profits = [6, 4, 5, 7]
capacity = 16

Para guardar una instancia del problema de la mochila en un solo objeto de Python podemos definir una **clase.**
Recordemos el siguiente ejemplo:



In [1]:
class NúmeroComplejo:
    def __init__(self, parte_real, parte_imaginaria):
        self.r = parte_real
        self.i = parte_imaginaria

z = NúmeroComplejo(2, -3)
z.r, z.i, z, type(z)

Entonces para una instancia del problema de la mochila podemos hacer:



In [1]:
class KnapsackProblem:
    def __init__(self, weights, profits, capacity):
        self.weights = weights
        self.profits = profits
        self.capacity = capacity

    def __repr__(self):
        return f"KP. Weights: {self.weights} Profits: {self.profits} Capacity: {self.capacity}"

prob = KnapsackProblem([5, 7, 3, 9], [6, 4, 5, 7], 15)
prob.weights, prob.profits, prob.capacity

In [1]:
prob

### Dataclasses



Una manera sencilla de definir una clase como ésta es definir una dataclass.



In [1]:
from dataclasses import dataclass

@dataclass
class KnapsackInstance:
    weights: list[int]
    profits: list[int]
    capacity: int

prob = KnapsackInstance([5, 7, 3, 9], [6, 4, 5, 7], 15)
prob.weights, prob.profits, prob.capacity

In [1]:
prob

## Primer solución



Necesitamos las siguientes funciones de Python:



### zip



In [1]:
parejas = zip([4, 5, 3], ["a", "b", "c"])
parejas

zip produce un iterador:



In [1]:
next(parejas)

### sum



In [1]:
sum((4, -1, 4, 2, 2))

In [1]:
def knapsack_solver(instance, idx=0, current=None):
    if current is None:
        current = []

    # Base Case: All items have been considered
    if idx == len(instance.weights):
        # Calculate weight and profit for this specific sequence
        total_weight = sum(w * s for w, s in zip(instance.weights, current))
        total_profit = sum(p * s for p, s in zip(instance.profits, current))
        
        # Only yield if it fits in the bag
        if total_weight <= instance.capacity:
            yield (total_profit, total_weight, current[:])
    else:
        for choice in [0, 1]:
            current.append(choice)
            yield from knapsack_solver(instance, idx + 1, current)
            current.pop()

In [1]:
prob = KnapsackInstance([5, 7, 3, 9], [6, 4, 5, 7], 16)
solutions = list(knapsack_solver(prob))
solutions

In [1]:
f = lambda x: x**3

f(2)

In [1]:
max([(4, 1), (2, 3), (2, 8), (4, 7)], key=(lambda x: x[0] + x[1]))

In [1]:
best_sol = max(solutions)

print(f"Best Profit: {best_sol[0]}")
print(f"Total Weight: {best_sol[1]}")
print(f"Selection (0/1): {best_sol[2]}")

In [1]:
def knapsack_solver(instance, idx=0, current=None, best=None):
    if current is None:
        current = []
    if best is None:
        # Initialize best with [profit, weight, selection]
        best = [0, 0, []]

    # Base Case: All items have been considered
    if idx == len(instance.weights):
        total_weight = sum(w * s for w, s in zip(instance.weights, current))
        total_profit = sum(p * s for p, s in zip(instance.profits, current))
        
        # Check if valid AND better than the current best
        if total_weight <= instance.capacity and total_profit > best[0]:
            best[0], best[1], best[2] = total_profit, total_weight, current[:]
            yield tuple(best) # Yield the new champion
    else:
        for choice in [1, 0]:
            current.append(choice)
            # Pass the same 'best' list down the recursion
            yield from knapsack_solver(instance, idx + 1, current, best)
            current.pop()

In [1]:
prob = KnapsackInstance([5, 7, 3, 9], [6, 4, 5, 7], 16)
sol = knapsack_solver(prob)
list(sol)

Aquí podemos hacer la observación de que si primero recorremos la opción 1 en lugar de la opción 0 podemos terminar más rápido.

Sin embargo, preferimos una función que regrese el resultado mejor solamente. En este caso, usaremos una función auxiliar y ya no usamos generadores, pero sí recursión. El estado de la mejor solución encontrada la guardamos en un diccionario.



In [1]:
def knapsack_solver(instance):
    best = {"profit": 0, "weight": 0, "selection": []}

    def solve(idx, current):
        if idx == len(instance.weights):
            total_weight = sum(w * s for w, s in zip(instance.weights, current))
            total_profit = sum(p * s for p, s in zip(instance.profits, current))
            
            if total_weight <= instance.capacity and total_profit > best["profit"]:
                best["profit"] = total_profit
                best["weight"] = total_weight
                best["selection"] = current[:]
        else:
            for choice in [1, 0]:
                current.append(choice)
                solve(idx + 1, current)
                current.pop()

    solve(0, [])
    return best

In [1]:
result = knapsack_solver(prob)
result

## Pruning



Aquí introducimos el concepto de "pruning". Si una rama del árbol no vale la pena seguir, la cortamos antes de generarla.

La función `solve` ahora ya tiene 3 parámetros, pues necesitamos llevar cuenta del peso parcial.



In [1]:
def knapsack_solver_prune(instance):
    best = {"profit": 0, "weight": 0, "selection": []}

    def solve(idx, current, current_weight):
        if idx == len(instance.weights):
            total_profit = sum(p * s for p, s in zip(instance.profits, current))
            if total_profit > best["profit"]:
                best["profit"] = total_profit
                best["weight"] = current_weight
                best["selection"] = current[:]
            return

        # Check if adding the next item (choice 1) exceeds capacity
        if current_weight + instance.weights[idx] <= instance.capacity:
            choices = [1, 0]
        else:
            choices = [0]

        for choice in choices:
            current.append(choice)
            # Update current_weight based on the choice made
            new_weight = current_weight + (instance.weights[idx] if choice == 1 else 0)
            solve(idx + 1, current, new_weight)
            current.pop()

    solve(0, [], 0)
    return best

In [1]:
result = knapsack_solver_prune(prob)
result

Comparemos resultados con una instancia más grande:



In [1]:
weights = [15, 12, 10, 8, 25, 30, 5, 18, 22, 14, 11, 9, 20, 13, 7, 24, 16, 19, 21, 6]
profits = [20, 15, 12, 10, 30, 35, 8, 22, 25, 18, 14, 11, 25, 17, 9, 28, 20, 23, 26, 8]
capacity = 40 

prob = KnapsackInstance(weights, profits, capacity)

In [1]:
knapsack_solver_prune(prob)